# 01 — The wisdom of trees: averaging cuts variance

*Chapter 06 — Random Forests · Notebook 1 of 5*

Welcome to the first ensemble of the course. Until now, every method has been a single model:
one set of neighbours, one line, one tree. Here we change the move entirely — we train **many**
models and let them vote. The reward is the one thing a single tree could not give us: **stability**.

**Prerequisites.**
- **Chapter 04 (Decision Trees)**, especially NB 4–5: a deep tree drives its training error to zero
  (it *memorizes*) and is **high-variance** — on `make_moons` its accuracy wobbled across resamples
  (NB 4), and on breast cancer its very first split changed from one patient sample to the next (NB 5).
  NB 5 ended by hand-bagging 25 trees and recovering much of the single-tree accuracy gap; this
  notebook explains why that works.
- **Module 00:** the train/test split (NB 04) and accuracy (NB 06).

**What you'll be able to do.**
- Draw a **bootstrap** resample of a dataset by hand, and say what it keeps and what it drops.
- Aggregate many trees by **majority vote** into one ensemble.
- *Measure* how averaging cuts variance — and read the diminishing returns.
- Explain the σ²/B intuition behind variance reduction.
- Recognize that the bagging you build by hand **is** a random forest with one idea switched off.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0
rng = np.random.default_rng(SEED)

# A 2-D, non-linear set — the same one chapter 04 carved, so the single-tree
# number is a continuity anchor (and 2-D means we can *draw* the variance).
X, y = make_moons(n_samples=300, noise=0.30, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape[0]} points    test: {X_test.shape[0]} points")
print(f"two features (x1, x2); two classes {np.unique(y).tolist()}")

## Where we are

Chapter 04 left us with a tension. A decision tree, grown deep, is wonderfully flexible — it can
carve any shape out of the feature space. But that same flexibility makes it **high-variance**: it
fits the training sample so closely that a small change in the data redraws the boundary. We saw the
root question itself flip when we resampled the patients, and on `make_moons` the test accuracy
wobbled with a standard deviation around 0.03.

A wobbly-but-flexible model is exactly the raw material an **ensemble** loves. The plan of this
notebook: keep the deep tree's flexibility, but average away its wobble by training many trees and
voting. We stay on `make_moons` for one reason — it is two-dimensional, so we can *draw* the variance
and watch it shrink.

## One tree is a shaky expert

Picture a single deep tree as an expert who has memorized one textbook cover to cover. Ask a question
from inside the book and the answer is razor-sharp; nudge the question slightly and the answer can
swing, because the expert latched onto the exact wording rather than the underlying idea. Train the
same tree on a slightly different sample and its decision boundary moves — sometimes a lot.

The cure is the *wisdom of crowds*: instead of trusting one memorizing expert, assemble a **committee**
of them, each having studied a slightly different version of the material, and let them **vote**. Where
one expert's quirks pull the answer astray, the others outvote the quirk. The signal they share
survives; the noise they each invented cancels.

A committee needs two ingredients: a way to give each tree a slightly different training set (the
**bootstrap**), and a way to combine their answers (the **majority vote**). We build both by hand.

In [ ]:
# One deep tree, then the same tree refit on many bootstrap resamples of the train set.
single = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
train_acc = accuracy_score(y_train, single.predict(X_train))
test_acc = accuracy_score(y_test, single.predict(X_test))
print(f"single deep tree:  train {train_acc:.3f}   test {test_acc:.3f}")

n = X_train.shape[0]
boot_rng = np.random.default_rng(SEED)
boot_acc = []
for _ in range(50):
    idx = boot_rng.integers(0, n, n)  # a bootstrap resample of the training set
    tree_b = DecisionTreeClassifier(random_state=SEED).fit(X_train[idx], y_train[idx])
    boot_acc.append(accuracy_score(y_test, tree_b.predict(X_test)))
boot_acc = np.array(boot_acc)
print(f"refit on 50 bootstraps: test mean {boot_acc.mean():.3f}   std {boot_acc.std():.3f}")

**Read the result.** The single tree scores **0.878** on the test set — respectable. But that number
is *fragile*: refit the very same tree on 50 bootstrap resamples of the training data and the test
accuracy swings with a standard deviation around **0.031** — roughly 0.85 to 0.92 depending on the
luck of the draw. The model's flexibility comes bundled with instability. The question for the rest of
this notebook: can we keep the flexibility and remove the wobble?

## Ingredient one: the bootstrap

A **bootstrap sample** is a fresh dataset drawn from the one we have, the same size, **sampling with
replacement**. We pick a training point at random, write it down, put it back, and repeat n times.
Because we put each point back, some points get picked twice or three times, and — this is the
interesting part — some never get picked at all.

How many are left out? Each single draw misses a given point with probability (1 − 1/n), and we draw n
times, so the chance a point is never picked is (1 − 1/n)ⁿ, which settles near **1/e ≈ 0.37** as n
grows. So a bootstrap sample holds about 63% of the distinct points (some duplicated) and leaves
roughly **37% on the bench**. Those benched points will earn their keep in notebook 3.

Each bootstrap is a plausible *alternate* dataset — the data we might have collected on a different
day. Train one tree per bootstrap and the committee is built.

In [ ]:
# Ten labelled points with ids 0..9 — draw one bootstrap and see what it keeps and drops.
ids = np.arange(10)
draw = rng.integers(0, 10, 10)
drawn, counts = np.unique(draw, return_counts=True)
omitted = np.setdiff1d(ids, drawn)
print(f"bootstrap draw (with replacement): {np.sort(draw).tolist()}")
print(f"picked more than once: {drawn[counts > 1].tolist()}")
print(f"never picked (out-of-bag): {omitted.tolist()}  ->  {len(omitted)}/10 left out")

# Averaged over many draws, the left-out fraction matches (1 - 1/n)^n; 1/e is its large-n limit.
left_out = [(10 - len(np.unique(rng.integers(0, 10, 10)))) / 10 for _ in range(5000)]
print(f"average left-out fraction (n=10): {np.mean(left_out):.3f}")
print(f"formula (1 - 1/10)**10 = {(1 - 1/10)**10:.3f}    1/e = {1/np.e:.3f} (the n -> inf limit)")

**Read it.** This resample kept some points more than once and dropped others entirely. Averaged over
many draws, the empirical left-out fraction (0.346 here) sits close to the closed form (1 − 1/n)ⁿ =
0.349 for n = 10, the small gap being sampling noise — and both climb toward **1/e ≈ 0.37** as n grows
(for our 210-point training set the closed form is already 0.367). A bootstrap is a slightly different
*view* of the same problem; grow a tree on each of many such views and no two trees see quite the same
data, which is precisely what we want.

## Ingredient two: the majority vote

With a committee of trees in hand, the **ensemble's** prediction is the class that **most trees vote
for** (for probabilities, we average their votes). That is the whole aggregation rule.

Why should voting help? Suppose each tree is right more often than not and — this is the crucial part —
their mistakes are *different* from one another. Then on any given point the wrong votes are scattered
while the right votes pile up, and the majority lands on the correct class far more reliably than any
single tree. The more the trees disagree in their errors, the more the vote cleans them up. We will
make this precise with the σ²/B law shortly; first, let's build it and watch it work.

In [ ]:
class HandBag:
    '''A bag of deep trees, each on its own bootstrap; predict = majority vote.'''

    def __init__(self, n_trees, seed=SEED):
        self.n_trees = n_trees
        self.seed = seed

    def fit(self, X, y):
        n = X.shape[0]
        boot_rng = np.random.default_rng(self.seed)
        self.trees_ = []
        for b in range(self.n_trees):
            idx = boot_rng.integers(0, n, n)
            self.trees_.append(DecisionTreeClassifier(random_state=b).fit(X[idx], y[idx]))
        return self

    def predict(self, X):
        votes = np.mean([tree.predict(X) for tree in self.trees_], axis=0)
        return (votes >= 0.5).astype(int)  # majority vote; even-B ties -> class 1 (>= 0.5)


for n_trees in [1, 5, 25, 100]:
    bag = HandBag(n_trees).fit(X_train, y_train)
    print(f"bagged {n_trees:3d} trees: test {accuracy_score(y_test, bag.predict(X_test)):.3f}")

**Read the result.** From a single tree's **0.878** to **0.933** with 25 trees — averaging recovered
about five points of accuracy, then flattened out. The committee beats its members. But accuracy is
only half the story, and not the more important half: the real prize is **stability**. The next two
figures show both — first what the variance *looks* like, then how fast it shrinks.

In [ ]:
# Five single trees (each on its own bootstrap) vs the committee of 100 — same test points beneath.
fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.5))
boot_rng = np.random.default_rng(SEED)
for i in range(5):
    idx = boot_rng.integers(0, n, n)
    tree_b = DecisionTreeClassifier(random_state=i).fit(X_train[idx], y_train[idx])
    viz.plot_decision_boundary(tree_b, X_test, y_test, resolution=200, ax=axes.flat[i])
    axes.flat[i].set_title(f"single tree, bootstrap #{i + 1}")
    axes.flat[i].legend().remove()

bag100 = HandBag(100).fit(X_train, y_train)
viz.plot_decision_boundary(bag100, X_test, y_test, resolution=200, ax=axes.flat[5])
axes.flat[5].set_title("bagged: 100 trees, majority vote")
axes.flat[5].legend().remove()
fig.tight_layout()

**Read the figure.** Each of the first five panels is one tree trained on one bootstrap resample.
Notice two things: every boundary is **jagged** (the tree fenced off individual points), and every
boundary is **different** from its neighbours — that difference, panel to panel, *is* the variance we
measured. The last panel is the committee of 100 trees voting. Its boundary is **smooth**: the
idiosyncratic spikes each tree invented have been outvoted, and only the shape they agree on remains.
Averaging did not make any single tree smarter — it made the *committee* steadier than any member.

In [ ]:
# Mean test accuracy and run-to-run std across 20 independent bagging seeds, vs number of trees.
tree_counts = [1, 2, 3, 5, 10, 25, 50, 100]
mean_acc, std_acc = [], []
for n_trees in tree_counts:
    accs = [accuracy_score(y_test, HandBag(n_trees, seed=s).fit(X_train, y_train).predict(X_test))
            for s in range(20)]
    mean_acc.append(np.mean(accs))
    std_acc.append(np.std(accs))

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(tree_counts, mean_acc, marker="o", color=COLORS["model"], label="test accuracy (mean)")
ax2.plot(tree_counts, std_acc, marker="s", color=COLORS["highlight"], label="run-to-run std")
ax1.set_xscale("log")
ax1.set_xlabel("number of trees (B)")
ax1.set_ylabel("test accuracy")
ax2.set_ylabel("run-to-run std of test accuracy")
ax2.grid(False)
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [ln.get_label() for ln in lines], loc="center right")
fig.tight_layout()
print("mean accuracy:", [round(float(a), 3) for a in mean_acc])
print("run-to-run std:", [round(float(s), 4) for s in std_acc])

**Read the figure.** Two stories share one plot. The accuracy curve climbs quickly and then
**plateaus** — past roughly 25 trees, adding more barely moves the mean (the *diminishing returns* of
averaging). The other curve is the one to watch: the run-to-run standard deviation falls about
**nine-fold**, from ~0.047 with a single tree to ~0.005 with a hundred (the trend is clearly downward
though not perfectly smooth — 20 seeds is a modest sample, so a little wiggle is expected). The
ensemble's answer stops depending on the luck of the resample.

This is **variance reduction**, and it has a clean shape: averaging B noisy predictions shrinks their
spread roughly like σ²/B. More trees never cost us accuracy here — they only cost compute. (Why
"roughly" and not exactly σ²/B? Because the trees are correlated, not independent — the gap that the
next notebook closes.)

One caveat to carry forward: averaging cancels **variance**, not **bias**. It steadies a flexible,
low-bias learner like a deep tree; it cannot rescue a model class that is wrong for the data — a
committee of biased models stays biased. Bagging is a variance cure, and only that.

## You built bagging

The recipe — bootstrap the data, grow a tree on each resample, take the majority vote — has a name:
**bagging**, for **b**ootstrap **agg**regat**ing** (Breiman, 1996). You built it by hand, with nothing
but a tree and a random number generator.

A **random forest** is bagging plus exactly one more idea, which we meet in notebook 2. Before we add
it, let's check the thing we'd expect: does scikit-learn's forest, with that extra idea switched off,
reproduce our hand-built bag?

In [ ]:
hand = HandBag(200).fit(X_train, y_train)

# A forest with feature subsampling OFF (max_features=None) is exactly bagging.
rf_bag = RandomForestClassifier(n_estimators=200, max_features=None, random_state=SEED)
rf_bag.fit(X_train, y_train)
# The default forest keeps feature subsampling ON (max_features='sqrt').
rf_default = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_train, y_train)

acc_hand = accuracy_score(y_test, hand.predict(X_test))
acc_bag = accuracy_score(y_test, rf_bag.predict(X_test))
acc_def = accuracy_score(y_test, rf_default.predict(X_test))
print(f"our hand-bag (200 trees):                  test {acc_hand:.4f}")
print(f"RandomForestClassifier(max_features=None): test {acc_bag:.4f}")
print(f"RandomForestClassifier() default (sqrt):   test {acc_def:.4f}")

**Read the parity.** The library is not magic. `RandomForestClassifier(max_features=None)` reaches the
**same** converged accuracy as the bag we built by hand — because, with `max_features=None`, that is
exactly what it is: bootstrap, grow trees, vote.

The *default* forest scores a touch lower here (0.900). That is not a bug — it is the one extra idea,
**feature subsampling**, which the default turns on. On this two-feature toy it has little room to help
(and can even hurt). On real data with many features it is the difference between a bag of trees and a
true random *forest* — which is exactly where notebook 2 begins.

## Your turn

1. **Vote by hand (easy).** Three trees classify a point as `1`, `0`, `1`. What does the majority-vote
   ensemble predict? If a fourth tree votes `0`, how would you break the resulting tie — and what does
   that suggest about using an even number of trees?

2. **Bootstrap by hand (medium).** Take the ten points `0..9`. Draw a bootstrap with
   `rng.integers(0, 10, 10)`, list which ids repeat and which are missing, and compute the left-out
   fraction. Repeat a few times — does it hover near 0.37?

3. **Find the plateau (harder).** Sweep `n_trees` over `[1, 2, 5, 10, 25, 50, 100]`. For each, fit the
   hand-bag under several different seeds and record both the mean test accuracy and its standard
   deviation. Plot them. Where do the diminishing returns set in — and which curve, accuracy or std,
   keeps improving the longest?

## What you built

- You drew **bootstrap** resamples by hand and saw the ~37% out-of-bag fraction fall out of the 1/e
  argument.
- You aggregated deep trees by **majority vote** into an **ensemble** — building your own `HandBag`
  estimator with `fit` and `predict`.
- You *measured* **variance reduction**: a single tree's run-to-run std of ~0.031 shrank about
  nine-fold under averaging, while accuracy rose from 0.878 to 0.933.
- You read the **diminishing returns** of adding trees, and confirmed your hand-built bag matches
  `RandomForestClassifier(max_features=None)` — bagging is a forest with one idea switched off.

**Vocabulary you now own:** ensemble · bootstrap (sampling with replacement) · bagging (bootstrap
aggregating) · majority vote · variance reduction · diminishing returns.

## Going further (optional)

Here is the σ²/B idea with a little more care. Suppose we average B predictions, each with variance σ².
If they were perfectly **independent**, the average would have variance σ²/B — drive B up and the
variance vanishes. Real bagged trees are *not* independent: they are grown from overlapping bootstraps
of the same data, so they tend to make correlated mistakes. Writing ρ for the average pairwise
correlation, the variance of the average is

$$\operatorname{Var} = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2.$$

The second term melts away as B grows, but the first term, ρ·σ², is a **floor** that more trees cannot
break. Lowering that floor means making the trees *less correlated* — the missing ingredient, and the
whole subject of notebook 2.

## References

- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140.
  DOI: [10.1007/BF00058655](https://doi.org/10.1007/BF00058655)
- Breiman, L. (2001). *Random Forests.* Machine Learning 45, 5–32.
  DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §8.7 (bagging) and §15 (random forests).
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
  (2nd ed.), §8.2. DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: Chapter 05 — Support Vector Machines (NB 5: breast cancer). Next: 02 — The "random" in the
forest: decorrelating the trees.*